# MakeMyTrip Hotel & Travel Business Analytics
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** Hotel listings for Bangalore, Chennai, Delhi, Hyderabad, Kolkata and Mumbai.

## Project objective
Clean and explore the supplied hotel-listing data to compare hotel prices, ratings, review counts, star categories and city-level patterns. The notebook also generates business observations that are supported by the available columns.

> **Data limitation:** These files contain hotel listing information, not actual booking transactions. They do not establish bookings, cancellations, customer behavior, occupancy or MakeMyTrip revenue. Do not interpret listing price as realized revenue.

## 1. Import libraries
Run the next cell. If a library is missing, install it in the VS Code notebook terminal with `python -m pip install pandas numpy matplotlib seaborn jupyter`.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Locate the project data
This cell works if you open the notebook from the project root or from the `notebooks` folder. Keep the original CSVs in `data/raw/`.

In [ ]:
possible_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in possible_roots if (p / "data" / "raw").exists()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw. Extract the project ZIP and open the notebook inside "
        "MakeMyTrip_Travel_Analytics_Project/notebooks/."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
GRAPH_DIR = PROJECT_ROOT / "graphs"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DIR}")

print("Project folder:", PROJECT_ROOT)
print("CSV files found:", len(csv_files))
for file in csv_files:
    print("-", file.name)

## 3. Load and combine the city-wise files
We add `City_Source` from each filename so the origin city remains available for comparisons.

In [ ]:
frames = []
for file in csv_files:
    temp = pd.read_csv(file)
    temp["City_Source"] = file.stem.replace("_", " ").title()
    temp["Source_File"] = file.name
    frames.append(temp)

raw_df = pd.concat(frames, ignore_index=True, sort=False)
print("Combined shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.sample(min(5, len(raw_df)), random_state=42))

## 4. Initial data understanding

In [ ]:
print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nData types:")
display(raw_df.dtypes.rename("Data Type").to_frame())
print("\nDataset information:")
raw_df.info()

## 5. Missing values and duplicate checks

In [ ]:
missing_report = pd.DataFrame({
    "Missing_Count": raw_df.isna().sum(),
    "Missing_Percent": (raw_df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percent", ascending=False)
display(missing_report)

exact_duplicates = raw_df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

## 6. Clean the dataset
- Standardize column names.
- Trim extra spaces in text fields.
- Convert `Price`, `Tax`, `Rating`, `Reviews` and `Star Rating` to numeric types.
- Parse distance into numeric value and unit where available.
- Preserve missing tax/distance values as missing; do not assume they are zero.
- Keep potential price outliers and flag them separately instead of deleting them automatically.

In [ ]:
df = raw_df.copy()

def standardize_column(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")

df.columns = [standardize_column(c) for c in df.columns]

text_columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in text_columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    missing_text = df[col].astype("string").str.strip().str.lower().isin(["", "n/a", "na", "nan"]).fillna(False)
    df.loc[missing_text, col] = np.nan

def to_numeric_clean(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False).str.replace("₹", "", regex=False).str.strip(),
        errors="coerce"
    )

for col in ["rating", "reviews", "star_rating", "price", "tax"]:
    if col in df.columns:
        df[col] = to_numeric_clean(df[col])

# Parse landmark distance, e.g. "2.4 km" or "930 m"
if "distance_to_landmark" in df.columns:
    dist_text = df["distance_to_landmark"].astype("string").str.strip().str.lower()
    df["distance_value"] = pd.to_numeric(
        dist_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False), errors="coerce"
    )
    df["distance_unit"] = dist_text.str.extract(r"(km|m)\b", expand=False)
    is_m = df["distance_unit"].eq("m").fillna(False).to_numpy(dtype=bool)
    is_km = df["distance_unit"].eq("km").fillna(False).to_numpy(dtype=bool)
    df["distance_km"] = np.select(
        [is_m, is_km],
        [df["distance_value"] / 1000, df["distance_value"]],
        default=np.nan
    )

# Standardize common city labels while preserving the source city field.
df["city_source"] = df["city_source"].astype("string").str.strip().str.title()
for col in ["hotel_name", "rating_description", "location", "nearest_landmark"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Drop exact duplicate records only; retain similar listings unless confirmed duplicates.
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicate rows removed: {before - len(df)}")
print("Cleaned shape:", df.shape)
display(df.head())

## 7. Validate the cleaned data

In [ ]:
validation = {
    "rows": len(df),
    "columns": len(df.columns),
    "duplicate_rows_remaining": int(df.duplicated().sum()),
    "missing_price": int(df["price"].isna().sum()) if "price" in df else None,
    "nonpositive_price": int((df["price"] <= 0).sum()) if "price" in df else None,
    "rating_outside_0_to_5": int((~df["rating"].between(0, 5) & df["rating"].notna()).sum()) if "rating" in df else None,
    "negative_reviews": int((df["reviews"] < 0).sum()) if "reviews" in df else None,
}
display(pd.Series(validation, name="Validation Result").to_frame())
print("Missing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing_Count"))

## 8. Save cleaned data
This creates a new cleaned file. The source CSV files in `data/raw/` are not modified.

In [ ]:
clean_path = CLEAN_DIR / "makemytrip_hotels_cleaned.csv"
df.to_csv(clean_path, index=False)
print("Cleaned dataset saved to:", clean_path)
print("Rows saved:", len(df))

## 9. Descriptive statistics
The price column is the listed hotel price in the source data. The source does not specify enough context here to assume every listing uses identical dates, room occupancy or booking conditions.

In [ ]:
numeric_cols = [c for c in ["rating", "reviews", "star_rating", "price", "tax", "distance_km"] if c in df.columns]
display(df[numeric_cols].describe().T)

## 10. City-wise overview

In [ ]:
city_summary = df.groupby("city_source", dropna=False).agg(
    hotel_listings=("hotel_name", "count"),
    average_listed_price=("price", "mean"),
    median_listed_price=("price", "median"),
    average_rating=("rating", "mean"),
    average_reviews=("reviews", "mean"),
    average_star_rating=("star_rating", "mean"),
    listings_with_tax=("tax", "count")
).sort_values("hotel_listings", ascending=False).round(2)

display(city_summary)
city_summary.to_csv(CLEAN_DIR / "city_summary.csv")

## 11. Visual analysis
Each chart is also saved as a PNG in the `graphs/` folder.

In [ ]:
def save_chart(filename):
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

### 11.1 Number of hotel listings by source city

In [ ]:
plt.figure(figsize=(10, 5))
order = df["city_source"].value_counts().index
sns.countplot(data=df, y="city_source", order=order, color="steelblue")
plt.title("Hotel Listings by City Source")
plt.xlabel("Number of listings")
plt.ylabel("City source")
save_chart("01_listings_by_city.png")

### 11.2 Average listed price by city

In [ ]:
price_by_city = df.groupby("city_source", as_index=False)["price"].mean().sort_values("price", ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=price_by_city, x="price", y="city_source", color="steelblue")
plt.title("Average Listed Hotel Price by City")
plt.xlabel("Average listed price (source currency/units)")
plt.ylabel("City source")
save_chart("02_average_price_by_city.png")
display(price_by_city.round(2))

### 11.3 Price distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["price"].dropna(), bins=30, kde=True, color="steelblue")
plt.title("Distribution of Listed Hotel Prices")
plt.xlabel("Listed price")
plt.ylabel("Number of listings")
save_chart("03_price_distribution.png")

### 11.4 Price variation by city (box plot)

In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=df, x="city_source", y="price")
plt.title("Listed Price Spread by City")
plt.xlabel("City source")
plt.ylabel("Listed price")
plt.xticks(rotation=25, ha="right")
save_chart("04_price_by_city_boxplot.png")

### 11.5 Rating distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["rating"].dropna(), bins=15, discrete=False, color="steelblue")
plt.title("Distribution of Hotel Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of listings")
save_chart("05_rating_distribution.png")

### 11.6 Average rating by city

In [ ]:
rating_by_city = df.groupby("city_source", as_index=False)["rating"].mean().sort_values("rating", ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=rating_by_city, x="rating", y="city_source", color="steelblue")
plt.title("Average Hotel Rating by City")
plt.xlabel("Average rating")
plt.ylabel("City source")
save_chart("06_average_rating_by_city.png")
display(rating_by_city.round(2))

### 11.7 Relationship between price and rating

In [ ]:
plot_df = df.dropna(subset=["price", "rating"])
plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_df, x="rating", y="price", hue="city_source", alpha=0.75)
plt.title("Listed Price vs Hotel Rating")
plt.xlabel("Hotel rating")
plt.ylabel("Listed price")
plt.legend(title="City", bbox_to_anchor=(1.02, 1), loc="upper left")
save_chart("07_price_vs_rating.png")

### 11.8 Star category and price

In [ ]:
star_price = df.dropna(subset=["star_rating", "price"]).copy()
plt.figure(figsize=(9, 5))
sns.boxplot(data=star_price, x="star_rating", y="price")
plt.title("Listed Price by Star Rating")
plt.xlabel("Star rating")
plt.ylabel("Listed price")
save_chart("08_price_by_star_rating.png")

### 11.9 Reviews and ratings

In [ ]:
review_rating = df.dropna(subset=["reviews", "rating"])
plt.figure(figsize=(9, 6))
sns.scatterplot(data=review_rating, x="reviews", y="rating", hue="city_source", alpha=0.75)
plt.title("Number of Reviews vs Hotel Rating")
plt.xlabel("Number of reviews")
plt.ylabel("Rating")
plt.legend(title="City", bbox_to_anchor=(1.02, 1), loc="upper left")
save_chart("09_reviews_vs_rating.png")

### 11.10 Correlation heatmap

In [ ]:
corr_cols = [c for c in ["rating", "reviews", "star_rating", "price", "tax", "distance_km"] if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Between Numeric Variables")
save_chart("10_correlation_heatmap.png")
display(corr.round(2))

## 12. Potential price outliers (flag only)
An unusually high price is not automatically an error; it may reflect a luxury property. We use the IQR rule to flag potential outliers for review and keep them in the cleaned dataset.

In [ ]:
q1 = df["price"].quantile(0.25)
q3 = df["price"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df["price_outlier_flag"] = df["price"].notna() & (
    (df["price"] < lower_bound) | (df["price"] > upper_bound)
)
print(f"IQR lower bound: {lower_bound:,.2f}")
print(f"IQR upper bound: {upper_bound:,.2f}")
print("Potential price outliers:", int(df["price_outlier_flag"].sum()))
display(df.loc[df["price_outlier_flag"], ["hotel_name", "city_source", "star_rating", "rating", "price"]]
        .sort_values("price", ascending=False).head(20))
# Save final cleaned file including outlier review flag.
df.to_csv(clean_path, index=False)

## 13. Business insights generated from this dataset

In [ ]:
# Generate concise, data-derived observations without claiming actual bookings or revenue.
valid_price = df.dropna(subset=["price"])
valid_rating = df.dropna(subset=["rating"])

print("BUSINESS OBSERVATIONS")
print("---------------------")
if not valid_price.empty:
    city_price = valid_price.groupby("city_source")["price"].median().sort_values(ascending=False)
    print(f"Highest median listed price city: {city_price.index[0]} ({city_price.iloc[0]:,.2f})")
    print(f"Lowest median listed price city: {city_price.index[-1]} ({city_price.iloc[-1]:,.2f})")
    print(f"Overall median listed price: {valid_price['price'].median():,.2f}")
if not valid_rating.empty:
    city_rating = valid_rating.groupby("city_source")["rating"].mean().sort_values(ascending=False)
    print(f"Highest average rating city: {city_rating.index[0]} ({city_rating.iloc[0]:.2f}/5)")
if "star_rating" in df and "price" in df:
    star_medians = df.groupby("star_rating")["price"].median().dropna().sort_index()
    print("\nMedian listed price by star rating:")
    display(star_medians.to_frame("Median Listed Price"))
print("\nInterpretation reminder: these describe the supplied listings only; they are not booking volume or revenue results.")

## 14. Export a business summary for Power BI
The following city-level summary can be imported into Power BI alongside the cleaned listing-level data.

In [ ]:
powerbi_summary = df.groupby("city_source", as_index=False).agg(
    hotel_listings=("hotel_name", "count"),
    average_listed_price=("price", "mean"),
    median_listed_price=("price", "median"),
    minimum_listed_price=("price", "min"),
    maximum_listed_price=("price", "max"),
    average_rating=("rating", "mean"),
    total_reviews=("reviews", "sum"),
    average_star_rating=("star_rating", "mean"),
    potential_price_outliers=("price_outlier_flag", "sum")
)
powerbi_path = CLEAN_DIR / "powerbi_city_summary.csv"
powerbi_summary.to_csv(powerbi_path, index=False)
display(powerbi_summary.round(2))
print("Power BI summary saved to:", powerbi_path)

## 15. Recommended Power BI dashboard
Create these visuals after importing `makemytrip_hotels_cleaned.csv` and `powerbi_city_summary.csv`:

1. **KPI cards:** Hotel listings, average listed price, median listed price, average rating.
2. **Bar chart:** Average listed price by city source.
3. **Bar chart:** Average rating by city source.
4. **Scatter chart:** Rating vs listed price; use city source as legend.
5. **Box plot or column chart:** Listed price by star rating (a custom visual may be needed for box plots).
6. **Table:** Hotel name, city source, location, star rating, rating, reviews, price and tax.
7. **Slicers:** City source, star rating, rating description and location.

### Business questions the dashboard can answer
- Which city has the highest median listed hotel price?
- How do listed prices vary by star rating?
- Which cities have higher average ratings in this sample?
- Are higher ratings associated with higher listed prices?
- Which listings should be reviewed as potential price outliers?

### What additional data is needed for booking analysis?
To analyze actual travel bookings, add a booking-level dataset with fields such as booking ID, booking date, travel/check-in date, service type, destination, listed/paid amount, booking status, cancellation status and payment status. Do not invent these fields in the current dataset.

## 16. Final checklist
- [ ] Run all cells from top to bottom.
- [ ] Check `data/cleaned/makemytrip_hotels_cleaned.csv`.
- [ ] Review potential outliers instead of deleting them automatically.
- [ ] Review saved charts in `graphs/`.
- [ ] Import the cleaned data into Power BI.
- [ ] Add a README explaining the data limitation: hotel listings are not transaction records.